# Step 8: Train Multi-Horizon Models
## NEW: Train separate models at different process completion points

This notebook trains a CatBoostClassifier at each horizon k, using only features from process steps 1 through k. This quantifies how early in the process we can reliably detect outlier wafers.

In [1]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

start_time = time.time()
print("=" * 60)
print("Step 8: Training Multi-Horizon Models")
print("=" * 60)

Step 8: Training Multi-Horizon Models


In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")
print(f"\nData directory exists: {(Path.cwd() / 'data').exists()}")
print(f"Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline

Data directory exists: True
Outputs directory exists: True


In [3]:
# Setup paths
data_dir = Path("data")
output_dir = Path("outputs")
features_dir = output_dir / "features"
models_dir = output_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

sentinel = models_dir / "08_horizons.done"
force = False  # Set to True to force rebuild

if sentinel.exists() and not force:
    print(f"[OK] Horizon models already trained (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print("Training horizon models...")

[OK] Horizon models already trained (found outputs\models\08_horizons.done)
  Set force=True to rebuild


In [4]:
# Check required files
required_files = [
    features_dir / "train.parquet",
    features_dir / "val.parquet",
    features_dir / "column_step_map.json"
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(
            f"Required file not found: {file}\n"
            f"Please run previous pipeline steps (especially steps 02, 03, 05)."
        )

In [5]:
# Load datasets
print("\nLoading training and validation data...")
train_df = pd.read_parquet(features_dir / "train.parquet")
val_df = pd.read_parquet(features_dir / "val.parquet")

print(f"  Train size: {len(train_df)}")
print(f"  Val size: {len(val_df)}")


Loading training and validation data...
  Train size: 13510
  Val size: 3307


In [6]:
# Load column-to-step mapping
print("\nLoading column-step mapping...")
with open(features_dir / "column_step_map.json", 'r') as f:
    column_step_map = json.load(f)

print(f"  Loaded mappings for {len(column_step_map)} features")
print(f"  SeqNo range: {min(column_step_map.values())} to {max(column_step_map.values())}")


Loading column-step mapping...
  Loaded mappings for 10302 features
  SeqNo range: 0 to 240


In [7]:
# Load step sequence to define horizons
print("\nDefining horizons...")
step_seq_file = data_dir / "step_seq.csv"

if not step_seq_file.exists():
    print(f"  WARNING: {step_seq_file} not found.")
    print(f"  Using SeqNo values from column_step_map instead.")
    all_seqnos = sorted(set(column_step_map.values()))
    all_seqnos = [s for s in all_seqnos if s > 0]  # Exclude SeqNo=0
else:
    step_seq_df = pd.read_csv(step_seq_file)
    all_seqnos = sorted(step_seq_df['SeqNo'].unique())
    print(f"  Loaded {len(all_seqnos)} step sequence numbers")


Defining horizons...
  Loaded 240 step sequence numbers


In [8]:
# Define horizons at deciles (10%, 20%, ..., 100% of process)
percentiles = np.percentile(all_seqnos, [10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
horizons = sorted(set([int(p) for p in percentiles]))

print(f"\n  Training models at {len(horizons)} horizons:")
for h in horizons:
    pct = 100 * h / max(all_seqnos)
    print(f"    Horizon {h:>4} ({pct:>5.1f}% complete)")


  Training models at 10 horizons:
    Horizon   24 ( 10.0% complete)
    Horizon   48 ( 20.0% complete)
    Horizon   72 ( 30.0% complete)
    Horizon   96 ( 40.0% complete)
    Horizon  120 ( 50.0% complete)
    Horizon  144 ( 60.0% complete)
    Horizon  168 ( 70.0% complete)
    Horizon  192 ( 80.0% complete)
    Horizon  216 ( 90.0% complete)
    Horizon  240 (100.0% complete)


In [9]:
# Prepare metadata columns
metadata_cols = ['WAFER_SCRIBE', 'LOT_ID', 'is_outlier', 'PARAM_END_DATETIME', 
                'first_start_time']
feature_cols = [col for col in train_df.columns if col not in metadata_cols]

print(f"\n  Total available features: {len(feature_cols)}")


  Total available features: 10302


In [10]:
# Extract labels
y_train = train_df['is_outlier']
y_val = val_df['is_outlier']

# Compute class weight
scale_pos_weight = (1 - y_train).sum() / y_train.sum()
print(f"\n  Class balance (train): {y_train.mean():.4f} outliers")
print(f"  scale_pos_weight: {scale_pos_weight:.2f}")


  Class balance (train): 0.0173 outliers
  scale_pos_weight: 56.74


## Train Model at Each Horizon

For each horizon k, we:
1. Filter features to those with SeqNo ≤ k (plus SeqNo=0 which are always included)
2. Train a CatBoost model on the filtered features
3. Evaluate on validation set
4. Save model and metrics

In [11]:
# Train models at each horizon
horizon_results = []

print("\n" + "=" * 80)
print("Training horizon models...")
print("=" * 80)
print(f"{'Horizon':<10} {'Features':<10} {'ROC AUC':<10} {'PR AUC':<10} {'F1':<8} {'Precision':<12} {'Recall':<8}")
print("-" * 80)

for k in tqdm(horizons, desc="Horizons"):
    # Filter features to those available at horizon k
    available_features = [
        col for col in feature_cols 
        if col in column_step_map and column_step_map[col] <= k
    ]
    
    if len(available_features) == 0:
        print(f"\n  WARNING: No features available at horizon {k}, skipping...")
        continue
    
    # Prepare feature matrices
    X_train = train_df[available_features].copy()
    X_val = val_df[available_features].copy()
    
    # Identify categorical features in this subset
    cat_features = []
    for col in available_features:
        is_categorical = (
            X_train[col].dtype == 'object' or
            X_train[col].dtype.name == 'string' or
            str(X_train[col].dtype).startswith('str') or
            col.startswith('LOT_ID') or
            '__EQUIP' in col or
            '__POSITION' in col or
            col in ['first_step', 'last_step']
        )
        if is_categorical:
            cat_features.append(col)
            # Ensure categorical columns are strings
            X_train[col] = X_train[col].fillna('UNKNOWN').astype(str)
            X_val[col] = X_val[col].fillna('UNKNOWN').astype(str)
    
    # Create CatBoost pools
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    
    # Train model
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric='AUC',
        scale_pos_weight=scale_pos_weight,
        early_stopping_rounds=30,
        random_seed=42,
        verbose=0  # Suppress per-iteration output
    )
    
    model.fit(train_pool, eval_set=val_pool)
    
    # Evaluate
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    roc_auc = roc_auc_score(y_val, y_pred_proba)
    pr_auc = average_precision_score(y_val, y_pred_proba)
    f1 = f1_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    
    # Save model
    model_file = models_dir / f"horizon_{k:03d}_model.cbm"
    model.save_model(str(model_file))
    
    # Store results
    horizon_results.append({
        'horizon': int(k),
        'n_features': len(available_features),
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'f1': float(f1),
        'precision': float(precision),
        'recall': float(recall)
    })
    
    # Print summary line
    print(f"k={k:<7} {len(available_features):<10} {roc_auc:<10.3f} {pr_auc:<10.3f} "
          f"{f1:<8.2f} {precision:<12.3f} {recall:<8.3f}")

print("-" * 80)


Training horizon models...
Horizon    Features   ROC AUC    PR AUC     F1       Precision    Recall  
--------------------------------------------------------------------------------


Horizons:  10%|█         | 1/10 [00:09<01:24,  9.35s/it]

k=24      1766       0.566      0.020      0.02     0.012        0.038   


Horizons:  20%|██        | 2/10 [00:24<01:41, 12.72s/it]

k=48      4373       0.630      0.023      0.03     0.020        0.058   


...

k=192     9445       0.873      0.545      0.51     0.462        0.577   


Horizons:  90%|█████████ | 9/10 [04:44<00:36, 36.59s/it]

k=216     9847       0.896      0.585      0.39     0.277        0.635   


Horizons: 100%|██████████| 10/10 [05:53<00:00, 35.33s/it]

k=240     10302      0.887      0.526      0.55     0.667        0.462   
--------------------------------------------------------------------------------


In [12]:
# Save results to JSON
results_file = output_dir / "horizon_results.json"
with open(results_file, 'w') as f:
    json.dump(horizon_results, f, indent=2)

print(f"\n[OK] Saved horizon_results.json with {len(horizon_results)} entries")


[OK] Saved horizon_results.json with 10 entries


In [13]:
# Summary statistics
print("\n" + "=" * 60)
print("Horizon Model Training Summary")
print("=" * 60)
print(f"Total horizons evaluated: {len(horizon_results)}")
print(f"Horizon range: {min(r['horizon'] for r in horizon_results)} to "
      f"{max(r['horizon'] for r in horizon_results)}")
print(f"\nPerformance range:")
print(f"  ROC AUC: {min(r['roc_auc'] for r in horizon_results):.3f} to "
      f"{max(r['roc_auc'] for r in horizon_results):.3f}")
print(f"  PR AUC:  {min(r['pr_auc'] for r in horizon_results):.3f} to "
      f"{max(r['pr_auc'] for r in horizon_results):.3f}")
print(f"  F1:      {min(r['f1'] for r in horizon_results):.3f} to "
      f"{max(r['f1'] for r in horizon_results):.3f}")

# Find first viable prediction point (ROC AUC > 0.80)
viable_horizons = [r for r in horizon_results if r['roc_auc'] > 0.80]
if viable_horizons:
    first_viable = min(viable_horizons, key=lambda r: r['horizon'])
    max_seqno = max(r['horizon'] for r in horizon_results)
    pct_complete = 100 * first_viable['horizon'] / max_seqno
    print(f"\nFirst viable prediction point (ROC AUC > 0.80):")
    print(f"  Horizon {first_viable['horizon']} ({pct_complete:.1f}% complete)")
    print(f"  ROC AUC: {first_viable['roc_auc']:.3f}")
    print(f"  Features: {first_viable['n_features']}")


Horizon Model Training Summary
Total horizons evaluated: 10
Horizon range: 24 to 240

Performance range:
  ROC AUC: 0.566 to 0.896
  PR AUC:  0.020 to 0.585
  F1:      0.019 to 0.545

First viable prediction point (ROC AUC > 0.80):
  Horizon 96 (40.0% complete)
  ROC AUC: 0.860
  Features: 7249


In [14]:
# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Horizon model training complete in {elapsed:.2f}s")
print("=" * 60)


[OK] Horizon model training complete in 449.93s
